# MongoDB Basics: How to Write, Update & Delete Data

## Student Enrollment Tracker

**Difficulty: Beginner | ~35 min | Requires Lab 1**

*Lab 2 of 7 in the MongoDB Mastery series.*

In this lab, you will learn the full **CRUD write lifecycle** in MongoDB.

You will learn how to:
1. Connect to a MongoDB Atlas cluster
2. Update a single document with `$set`
3. Update many documents at once with `$inc`
4. Perform upserts to handle re-enrollments
5. Delete documents individually and in bulk
6. Generate a formatted summary report

In [ ]:
!pip install -qU "pymongo[srv,tls]==4.10.1" python-dotenv==1.0.1 certifi

This installs the MongoDB Python driver (`pymongo`) with `srv` (for `mongodb+srv://` URIs) and `tls` extras, `python-dotenv` for loading credentials, and `certifi` for up-to-date CA certificates.

### Step 1 — Connect to MongoDB

In [ ]:
import os
import certifi
from dotenv import load_dotenv
import pymongo

# Load the Atlas connection string from the .env file one directory up
load_dotenv("../.env")
uri = os.environ["MONGODB_URI"]

# Connect to the real MongoDB Atlas cluster
client = pymongo.MongoClient(uri, tlsCAFile=certifi.where())

# Access the database and collection
db = client["school_db"]
students = db["students"]

print("Connected to MongoDB Atlas")

`load_dotenv("../.env")` reads the `.env` file in the `MongoDB-Labs` folder and loads `MONGODB_URI` into the environment. `certifi.where()` points pymongo to a trusted CA certificate bundle, which ensures SSL works reliably on all platforms.

### Step 2 — Insert the Starting Dataset

In [ ]:
# Drop the collection first so re-running the notebook starts clean
students.drop()

# Same 25 student records from Lab 1 — the starting point for write operations
student_records = [
    {"name": "Alice Johnson",  "student_id": "STU001", "course": "Computer Science", "grade": 95, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Bob Smith",      "student_id": "STU002", "course": "Mathematics",      "grade": 78, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Charlie Brown",  "student_id": "STU003", "course": "Physics",          "grade": 55, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Diana Prince",   "student_id": "STU004", "course": "English",          "grade": 48, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Eve Torres",     "student_id": "STU005", "course": "Biology",          "grade": 88, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Frank Castle",   "student_id": "STU006", "course": "Mathematics",      "grade": 52, "enrollment_date": "2024-09-01", "status": "inactive"},
    {"name": "Grace Hopper",   "student_id": "STU007", "course": "Computer Science", "grade": 91, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Hank Pym",       "student_id": "STU008", "course": "Physics",          "grade": 73, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Ivan Petrov",    "student_id": "STU009", "course": "Computer Science", "grade": 84, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Julia Child",    "student_id": "STU010", "course": "English",          "grade": 90, "enrollment_date": "2024-09-02", "status": "graduated"},
    {"name": "Karl Marx",      "student_id": "STU011", "course": "Mathematics",      "grade": 67, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Laura Palmer",   "student_id": "STU012", "course": "Biology",          "grade": 45, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Marco Polo",     "student_id": "STU013", "course": "Physics",          "grade": 82, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Nina Simone",    "student_id": "STU014", "course": "English",          "grade": 76, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Oscar Wilde",    "student_id": "STU015", "course": "Mathematics",      "grade": 93, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Pia Zadora",     "student_id": "STU016", "course": "Biology",          "grade": 71, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Quincy Adams",   "student_id": "STU017", "course": "Computer Science", "grade": 87, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Rosa Parks",     "student_id": "STU018", "course": "English",          "grade": 58, "enrollment_date": "2024-09-02", "status": "inactive"},
    {"name": "Sam Wilson",     "student_id": "STU019", "course": "Physics",          "grade": 98, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Tina Turner",    "student_id": "STU020", "course": "Mathematics",      "grade": 79, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Uma Thurman",    "student_id": "STU021", "course": "Computer Science", "grade": 62, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Vera Wang",      "student_id": "STU022", "course": "Biology",          "grade": 85, "enrollment_date": "2024-09-01", "status": "graduated"},
    {"name": "Walt Disney",    "student_id": "STU023", "course": "Physics",          "grade": 56, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Xena Warrior",   "student_id": "STU024", "course": "English",          "grade": 94, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Yusuf Islam",    "student_id": "STU025", "course": "Computer Science", "grade": 70, "enrollment_date": "2024-09-01", "status": "active"},
]

result = students.insert_many(student_records)
print(f"Inserted {len(result.inserted_ids)} student records.")

Each dictionary is a **document** — MongoDB's equivalent of a row. `students.drop()` clears the collection before inserting so the notebook is safe to re-run without duplicating data.

### Step 3 — Single Update: Add a Scholarship Field

In [ ]:
# update_one() modifies the first document that matches the filter
# $set creates the "scholarship" field if it doesn't exist
result = students.update_one(
    {"name": "Alice Johnson"},
    {"$set": {"scholarship": "Dean's Award"}}
)

print(f"Updated: Alice Johnson — added scholarship = 'Dean's Award'")

`update_one()` updates **only the first** matching document. `$set` either creates a new field or overwrites an existing one — it never removes other fields.

### Step 4 — Bulk Increment: Bump Failing Grades by 5

In [ ]:
# Capture who's failing BEFORE the update
failing_ids = [s["student_id"] for s in students.find(
    {"grade": {"$lt": 60}}, {"_id": 0, "student_id": 1}
)]

# update_many() modifies ALL documents that match the filter
# $inc adds 5 to the grade field (no need to read the current value first)
result = students.update_many(
    {"grade": {"$lt": 60}},
    {"$inc": {"grade": 5}}
)

print(f"Updated {result.modified_count} failing student(s) — grades incremented by 5")

# Look up the SAME students by their captured IDs, not by re-filtering on grade
updated_failing = students.find(
    {"student_id": {"$in": failing_ids}}, {"_id": 0}
)

print("\n--- Updated Failing Students ---")
for s in updated_failing:
    print(f"{s['name']:<20} | {s['course']:<12} | Grade: {s['grade']}")

`$inc` is atomic — MongoDB reads the current value, adds the increment, and writes it back in one operation. This is safer than reading, modifying in Python, and writing back, because another process can't slip in a write between the read and the write.

### Step 5 — Upsert: Re-enroll a Withdrawn Student

In [ ]:
# upsert=True: if the filter matches, update it; if nothing matches, insert a new document
# Frank Castle (STU006) has status "inactive" — this changes it to "active"
result = students.update_one(
    {"student_id": "STU006"},
    {"$set": {"status": "active"}},
    upsert=True
)

if result.upserted_id:
    print(f"Upserted: new document inserted with _id = {result.upserted_id}")
else:
    print(f"Upserted: Frank Castle — status set to 'active'")

Without `upsert=True`, `update_one` does nothing if the filter matches nothing. With it, MongoDB inserts a new document combining the filter fields and the update fields. Here the filter matches an existing student, so it behaves like a normal update — but the pattern is essential when you're not sure if a record already exists.

### Step 6 — Delete: Remove Graduated Students

In [ ]:
# find_one_and_delete: finds one document, deletes it, and returns what it removed
graduated = students.find_one_and_delete(
    {"status": "graduated"},
    projection={"_id": 0, "name": 1, "status": 1}
)

if graduated:
    print(f"Deleted: {graduated['name']} (graduated)")

# delete_many() removes ALL remaining documents matching the filter
result = students.delete_many({"status": "graduated"})
print(f"Deleted {result.deleted_count} graduated student(s).")

`find_one_and_delete` is atomic — it finds and removes in one step, returning the deleted document so you can log what was removed. `delete_many` then cleans up any remaining matches in one call.

### Step 7 — Summary Report

In [ ]:
# Count remaining students
total = students.count_documents({})

# Count how many students have the "scholarship" field (added in Step 3)
updated = students.count_documents({"scholarship": {"$exists": True}})

# Status distribution
pipeline = [
    {"$group": {"_id": "$status", "count": {"$sum": 1}}},
    {"$sort": {"_id": 1}}
]
status_counts = {doc["_id"]: doc["count"] for doc in students.aggregate(pipeline)}

# Students still failing after the grade bump
still_failing = list(students.find(
    {"grade": {"$lt": 60}},
    {"_id": 0}
).sort("grade", 1))

print("         ENROLLMENT TRACKER SUMMARY")
print(f"\nTotal students remaining: {total}")
print(f"Students with scholarship field: {updated}")

print("\n--- Status Distribution ---")
for status, count in sorted(status_counts.items()):
    print(f"  {status}: {count}")

print("\n--- Still Failing (Grade < 60) ---")
for s in still_failing:
    print(f"  {s['name']} ({s['course']}) — Grade: {s['grade']}")

Re-collects the key metrics from each step and prints a formatted summary — the same kind of report an enrollment office would review at the end of a processing cycle.